# 🚀 Automated Evolution X Builder for Redmi Note 13 5G (gold/iron)
Notebook ini mengompilasi Evolution X (Android 15) untuk Redmi Note 13 5G / POCO X6 Neo secara otomatis.

### Cara Pakai (Tinggal 1 Klik):
1. Pastikan Runtime: Menu **Runtime** -> **Change runtime type** -> pilih **High-RAM**.
2. Klik menu **Runtime** -> **Run all** (atau tekan `Ctrl + F9`).
3. Saat Cell 1 minta izin sambung Google Drive, klik **Connect to Google Drive**.
4. Duduk santai, file zip ROM akan otomatis masuk ke Google Drive di folder `EvolutionX_gold/`!

In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('✓ Google Drive Terhubung!')

In [ ]:
%%bash
# Step 2: Install Android Build Dependencies & Repo Tool
set -e
sudo apt-get update -qq
sudo apt-get install -y -qq git-core gnupg flex bison build-essential zip curl zlib1g-dev \
libc6-dev-i386 libncurses5 x11proto-core-dev libx11-dev lib32z1-dev libgl1-mesa-dev \
libxml2-utils xsltproc unzip fontconfig python3 python-is-python3 rsync bc libssl-dev ccache git-lfs

mkdir -p /root/bin
curl -s https://storage.googleapis.com/git-repo-downloads/repo > /root/bin/repo
chmod a+x /root/bin/repo
ln -sf /root/bin/repo /usr/bin/repo

git config --global user.name "RizalNurfajri"
git config --global user.email "rizalnurfajri@users.noreply.github.com"
git config --global color.ui false
echo "✓ Build tools installed successfully!"

In [ ]:
%%bash
# Step 3: Sync Source Evolution X & Redmi Note 13 5G Trees
mkdir -p /content/rom && cd /content/rom

echo "==> Initializing Evolution X Manifest (vic / Android 15)..."
repo init -u https://github.com/Evolution-X/manifest -b vic --git-lfs --depth=1

echo "==> Injecting Device Tree gold/iron..."
mkdir -p .repo/local_manifests
cat << 'EOF' > .repo/local_manifests/gold.xml
<?xml version="1.0" encoding="UTF-8"?>
<manifest>
  <remote name="gold_src" fetch="https://github.com/Star-caorui" />
  <project name="android_device_xiaomi_gold" path="device/xiaomi/gold" remote="gold_src" revision="lineage-23.2" />
  <project name="android_device_xiaomi_gold-kernels" path="device/xiaomi/gold-kernel" remote="gold_src" revision="lineage-23.2" clone-depth="1" />
  <project name="android_vendor_xiaomi_gold" path="vendor/xiaomi/gold" remote="gold_src" revision="lineage-23.2" clone-depth="1" />
  <project name="LineageOS/android_device_mediatek_sepolicy_vndr" path="device/mediatek/sepolicy_vndr" />
  <project name="LineageOS/android_hardware_mediatek" path="hardware/mediatek" />
  <project name="LineageOS/android_hardware_xiaomi" path="hardware/xiaomi" />
</manifest>
EOF

echo "==> Downloading Sources (High-speed Google Backbone)..."
repo sync -c -j$(nproc --all) --force-sync --no-clone-bundle --no-tags

echo "==> Cleaning metadata .repo to save ~50GB disk space..."
rm -rf .repo
df -h
echo "✓ Sync completed!"

In [ ]:
%%bash
# Step 4: Inject Evolution X Makefiles for gold
cd /content/rom/device/xiaomi/gold

cat << 'EOF' > evolution_gold.mk
# Inherit Evolution X configs
$(call inherit-product, vendor/evolution/config/common_full_phone.mk)

# Inherit from device makefile
$(call inherit-product, device/xiaomi/gold/device.mk)

PRODUCT_NAME := evolution_gold
PRODUCT_DEVICE := gold
PRODUCT_MANUFACTURER := Xiaomi
PRODUCT_BRAND := Redmi
PRODUCT_MODEL := 2312DRAABC

TARGET_SUPPORTS_QUICK_TAP := true
EVO_BUILD_TYPE := UNOFFICIAL
EOF

cat << 'EOF' >> AndroidProducts.mk
PRODUCT_MAKEFILES += $(LOCAL_DIR)/evolution_gold.mk
COMMON_LUNCH_CHOICES += evolution_gold-userdebug
EOF
echo "✓ Makefiles configured!"

In [ ]:
%%bash
# Step 5: Start ROM Compilation
cd /content/rom
export USE_CCACHE=1
export CCACHE_DIR=/content/ccache
export CCACHE_EXEC=$(which ccache)
ccache -M 20G

source build/envsetup.sh
lunch evolution_gold-userdebug

echo "==> Compiling Evolution X for Redmi Note 13 5G (gold/iron)..."
m evolution -j$(nproc --all) || m bacon -j$(nproc --all)

echo "==> Copying output ZIP to Google Drive..."
mkdir -p /content/drive/MyDrive/EvolutionX_gold/
cp /content/rom/out/target/product/gold/evolution_gold*.zip /content/drive/MyDrive/EvolutionX_gold/ || true
cp /content/rom/out/target/product/gold/EvolutionX*.zip /content/drive/MyDrive/EvolutionX_gold/ || true

echo "✓ COMPILATION SELESAI! File ROM sudah tersimpan di Google Drive folder EvolutionX_gold/"